# Lab 05 - Policy Gradients on a Toy MDP

**Difficulty:** ★★☆ · **CORE** · est. build 1 day · Scaffold: stubs only. CPU-only (<1M-param char policy).

**Watch/read first**
- Lecture 3 (RL theory) — https://rlhfbook.com/course/
- Chapter 6 §§1–3 — https://rlhfbook.com/c/06-policy-gradients.html
- Gap-lab status: the repo jumps straight to LLM rollouts; this on-ramp is ours.

**Goal:** REINFORCE by hand on an inspectable task — emit `a+b` expressions evaluating to target N — then feel WHY variance is the practical problem: baseline on/off, K∈{2,4,16} leave-one-out, entropy collapse.

In [ ]:
# Setup -- CPU only, no GPU anywhere in this lab (see labs_curriculum/LABS_SETUP.md)
import json, math, random
from pathlib import Path
import numpy as np

import starter
CFG_PATH = Path("configs/05_policy_gradients_toy_mdp.yaml")
RUN_DIR = Path("runs/lab05_toy_mdp"); RUN_DIR.mkdir(parents=True, exist_ok=True)
seed = 42; random.seed(seed); np.random.seed(seed)
cfg_d = starter.load_config(str(CFG_PATH)); starter.validate_config(cfg_d)
cfg = starter.ToyMDPConfig.from_dict(cfg_d)
print("vocab:", len(starter.CHAR_ACTIONS), "| max chars:", starter.MAX_EPISODE_CHARS)

## Part 1 - Env & policy forward

TODO list:
1. `ToyArithmeticEnv.reset/step`: terminate on `'='` or 12 chars; reward = +1 iff `starter.expression_value(text) == target_n` else 0 (verifier spec is given plumbing you can read now)
2. `CharPolicyNet.init_params` + `logits_for_prefix`: keep total params < 1M (assert it); document your prefix-encoding scheme
3. Hand-write `softmax` / `log_softmax` with shift-by-max stability — no scipy/torch

In [ ]:
# TODO(student): implement env + net + distributions, then smoke-test one episode.
env = starter.ToyArithmeticEnv(cfg.target_range_min, cfg.target_range_max)
net = starter.CharPolicyNet(hidden_size=cfg.hidden_size)
params = net.init_params(np.random.default_rng(seed))          # NotImplementedError
trj = starter.sample_trajectory(params, net, env, np.random.default_rng(seed))
trj.validate()

## Part 2 - REINFORCE by hand (the from-scratch checkpoint)

Derivation to write out BEFORE coding: grad E[R] ≈ E[Σ_t ∇log π(aₜ|sₜ)·A(τ)]. The loss value: mean of −logπ(τ)·A.

Then variance reduction:
- `compute_advantages(mode='none')` — raw reward
- `mode='moving_avg'` — b = EMA of past mean rewards (α=0.9)
- `mode='loo'` — per group of K: Aᵢ = rᵢ − mean(r₋ᵢ)

Debugging exercises (README §debugging): name the collapse-to-repeated-action failure; trace why identical group rewards still give nonzero gradient.

In [ ]:
# TODO(student): implement losses/advantages and run E1/E2 experiments.
loss = starter.reinforce_loss([trj], np.array([trj.reward]))   # NotImplementedError
A_none = starter.compute_advantages([trj], mode="none")
H = starter.entropy_of_policy(starter.softmax(net.logits_for_prefix(params, 5, [])))

## Part 3 - Variance comparison experiment

E1: baseline none vs moving_avg vs loo. E2: K ∈ {2,4,16}.
Produce: gradient-variance plots (`gradient_variance_estimate` over stacked grads), success-rate curves, entropy traces (`observe_entropy_collapse`) — explain them line by line ("done when" criterion).

In [ ]:
# TODO(student): run_variance_experiment drives all arms; plot its artifacts here.
results = starter.run_variance_experiment(cfg, n_trials=20)     # NotImplementedError

## Part 4 - Write-up & compare

Done when: working REINFORCE + variance plots explained line-by-line; entropy-collapse diagnostic run; README questions answered (why log-prob not prob; what a baseline changes — and not; why variance is THE problem).

**Only now** open the answer key: `rlhf-book/code/policy_gradients/loss.py::ReinforceLoss` — diff conventions, not implementations.